
Pretrain

1. ЭТАП.
Скачайте данные из репозитория и упакуйте их в один датасет. Вам понадобятся все произведения из репозитория.

In [1]:
!git clone https://github.com/JoannaBy/RussianNovels.git

fatal: destination path 'RussianNovels' already exists and is not an empty directory.


In [2]:
!pip install pandas

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import os
import pandas as pd

folder = "RussianNovels/corpus"

data = []

for filename in os.listdir(folder):
    if filename.endswith(".txt"):
        
        file_path = os.path.join(folder, filename)
        
        with open(file_path, "r", encoding="utf-8") as file:
            text = file.read()
        
        data.append({
            "filename": filename,
            "text": text
        })

dataset = pd.DataFrame(data)

print("Количество произведений:", len(dataset))
dataset.head()

Количество произведений: 108


,filename,text
0,Pushkin_CapitanskaaDochka.txt,КАПИТАНСКАЯ ДОЧКА\n\nБереги честь смолоду.\n\n...
1,Sologub_KorolevaOrtruda.txt,ГЛАВА ТРИДЦАТЬ ЧЕТВЕРТАЯ \n\n \n О...
2,Chekhov_Dama.txt,Антон Павлович Чехов \nДама с собачкой\n \...
3,Gogol_Viy.txt,Как только ударял в Киеве поутру довольно звон...
4,Gorky_ZyznKlimaSamgina3.txt,"Дома, по комнатам тяжело носила изработанно..."


In [4]:
dataset.to_json(
    "russian_novels_dataset.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

print("Датасет сохранён!")

Датасет сохранён!


In [5]:
print(dataset.iloc[0]["filename"])
print(dataset.iloc[0]["text"][:500])

Pushkin_CapitanskaaDochka.txt
КАПИТАНСКАЯ ДОЧКА

Береги честь смолоду.

Пословица.

ГЛАВА I

СЕРЖАНТ ГВАРДИИ

-- Был бы гвардии он завтра ж капитан.

-- Того не надобно; пусть в армии послужит.

-- Изрядно сказано! пускай его потужит...

. . . . . . . . . . . . . . .

Да кто его отец?

Княжнин.

   Отец мой Андрей Петрович Гринев в молодости своей служил при графе Минихе и вышел в отставку премьер-майором в 17.. году. С тех пор жил он в своей Симбирской деревне, где и женился на девице Авдотье Васильевне Ю., дочери бедного т


In [6]:
import os

print("Репозиторий скачан:", os.path.exists("RussianNovels/corpus"))
print("Датасет сохранён:", os.path.exists("russian_novels_dataset.jsonl"))

Репозиторий скачан: True
Датасет сохранён: True


2 ЭТАП

In [7]:
%pip install -q transformers

Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd

dataset = pd.read_json(
    "russian_novels_dataset.jsonl",
    lines=True
)

print("Количество произведений:", len(dataset))
dataset.head()

Количество произведений: 108


,filename,text
0,Pushkin_CapitanskaaDochka.txt,КАПИТАНСКАЯ ДОЧКА\n\nБереги честь смолоду.\n\n...
1,Sologub_KorolevaOrtruda.txt,ГЛАВА ТРИДЦАТЬ ЧЕТВЕРТАЯ \n\n \n О...
2,Chekhov_Dama.txt,Антон Павлович Чехов \nДама с собачкой\n \...
3,Gogol_Viy.txt,Как только ударял в Киеве поутру довольно звон...
4,Gorky_ZyznKlimaSamgina3.txt,"Дома, по комнатам тяжело носила изработанно..."


In [ ]:
import re


def clean_text(text):
    text = str(text)


    text = text.replace("\xa0", " ")
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    text = re.sub(r"(?:\.\s*){2,}", "… ", text)
    text = re.sub(r"-{2,}", "—", text)


    text = re.sub(r"!{2,}", "!", text)
    text = re.sub(r"\?{2,}", "?", text)
    text = re.sub(r",{2,}", ",", text)
    text = re.sub(r";{2,}", ";", text)


    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)


    parts = re.split(r"(?<=[.!?…])\s+|\n+", text)

    cleaned_parts = []
    seen_parts = set()

    for part in parts:
        part = part.strip()

        if not part:
            continue


        if re.search(r"[A-Za-z]", part):
            continue


        if not re.search(r"[А-Яа-яЁё]", part):
            continue


        duplicate_key = re.sub(
            r"\s+",
            " ",
            part.lower()
        )


        if duplicate_key in seen_parts:
            continue

        seen_parts.add(duplicate_key)
        cleaned_parts.append(part)

    return "\n".join(cleaned_parts)

In [12]:
import os

print(os.path.exists("cleaned_russian_novels.jsonl"))

False


In [ ]:
import re


def clean_text_fast(text):
    text = str(text)

    text = text.replace("\xa0", " ")
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    cleaned_lines = []
    seen_lines = set()

    for line in text.splitlines():
        line = line.strip()

        if not line:
            continue


        line = re.sub(r"(?:\.\s*){2,}", "…", line)
        line = re.sub(r"-{2,}", "—", line)
        line = re.sub(r"!{2,}", "!", line)
        line = re.sub(r"\?{2,}", "?", line)
        line = re.sub(r",{2,}", ",", line)
        line = re.sub(r";{2,}", ";", line)


        line = re.sub(r"\s+", " ", line).strip()


        if re.search(r"[A-Za-z]", line):
            continue


        if not re.search(r"[А-Яа-яЁё]", line):
            continue


        duplicate_key = line.lower()

        if duplicate_key in seen_lines:
            continue

        seen_lines.add(duplicate_key)
        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)

In [14]:
%pip install -q tqdm

Note: you may need to restart the kernel to use updated packages.


In [15]:
from tqdm.auto import tqdm

tqdm.pandas()

dataset["clean_text"] = dataset["text"].progress_apply(
    clean_text_fast
)

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 108/108 [00:10<00:00, 10.42it/s]


In [16]:
dataset = dataset[
    dataset["clean_text"].str.strip().str.len() > 0
].copy()

dataset = dataset.drop_duplicates(
    subset=["clean_text"]
).reset_index(drop=True)

print("Произведений после очистки:", len(dataset))

Произведений после очистки: 107


In [17]:
example = dataset[
    dataset["filename"].str.contains(
        "CapitanskaaDochka",
        case=False,
        na=False
    )
]

print(example.iloc[0]["clean_text"][:1500])

КАПИТАНСКАЯ ДОЧКА
Береги честь смолоду.
Пословица.
СЕРЖАНТ ГВАРДИИ
— Был бы гвардии он завтра ж капитан.
— Того не надобно; пусть в армии послужит.
— Изрядно сказано! пускай его потужит…
Да кто его отец?
Княжнин.
Отец мой Андрей Петрович Гринев в молодости своей служил при графе Минихе и вышел в отставку премьер-майором в 17…году. С тех пор жил он в своей Симбирской деревне, где и женился на девице Авдотье Васильевне Ю., дочери бедного тамошнего дворянина. Нас было девять человек детей. Все мои братья и сестры умерли во младенчестве.
Матушка была еще мною брюхата, как уже я был записан в Семеновский полк сержантом, по милости майора гвардии князя Б., близкого нашего родственника. Если бы паче всякого чаяния матушка родила дочь, то батюшка объявил бы куда следовало о смерти неявившегося сержанта, и дело тем бы и кончилось. Я считался в отпуску до окончания наук. В то время воспитывались мы не по-нонешнему. С пятилетнего возраста отдан я был на руки стремянному Савельичу, за трезвое пове

In [18]:
print(
    "Дубликатов произведений:",
    dataset["clean_text"].duplicated().sum()
)

Дубликатов произведений: 0


In [19]:
%pip install -q transformers

Note: you may need to restart the kernel to use updated packages.


In [15]:
%pip uninstall -y torch torchvision torchaudio nvidia-nccl-cu12

Found existing installation: torch 2.13.0
Uninstalling torch-2.13.0:
  Successfully uninstalled torch-2.13.0
Found existing installation: nvidia-nccl-cu12 2.21.5
Uninstalling nvidia-nccl-cu12-2.21.5:
  Successfully uninstalled nvidia-nccl-cu12-2.21.5
Note: you may need to restart the kernel to use updated packages.


In [16]:
%pip install --no-cache-dir --force-reinstall \
    torch==2.5.1 \
    torchvision==0.20.1 \
    torchaudio==2.5.1 \
    --index-url https://download.pytorch.org/whl/cu121

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 110.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 115.7 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 118.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 14.0 MB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 125.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 KB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 KB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 67.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 67.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 68.7 MB/s eta 0:00:0000:010

In [20]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "ai-forever/rugpt3small_based_on_gpt2"
)

tokenizer.add_special_tokens({
    "bos_token": "<bos>",
    "eos_token": "<eos>"
})

print("BOS:", tokenizer.bos_token)
print("EOS:", tokenizer.eos_token)

BOS: <bos>
EOS: <eos>


In [21]:
CONTEXT_LENGTH = 512
TEXT_LENGTH = CONTEXT_LENGTH - 2

chunks = []

for _, row in dataset.iterrows():

    token_ids = tokenizer.encode(
        row["clean_text"],
        add_special_tokens=False
    )

    chunk_id = 0

    for start in range(0, len(token_ids), TEXT_LENGTH):

        part_ids = token_ids[start:start + TEXT_LENGTH]

        if len(part_ids) == 0:
            continue

        final_ids = (
            [tokenizer.bos_token_id]
            + part_ids
            + [tokenizer.eos_token_id]
        )

        part_text = tokenizer.decode(
            part_ids,
            skip_special_tokens=True
        ).strip()

        chunks.append({
            "filename": row["filename"],
            "chunk_id": chunk_id,
            "text": f"<bos>{part_text}<eos>",
            "input_ids": final_ids,
            "n_tokens": len(final_ids)
        })

        chunk_id += 1

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (51568 > 2048). Running this sequence through the model will result in indexing errors
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [ ]:
import pandas as pd


chunks_dataset = pd.DataFrame(chunks)


chunks_dataset = chunks_dataset[
    ~chunks_dataset["input_ids"].apply(tuple).duplicated()
].reset_index(drop=True)

print("Количество чанков:", len(chunks_dataset))

Количество чанков: 20065


In [23]:
print("Дубликатов чанков:", 
      chunks_dataset["input_ids"].apply(tuple).duplicated().sum())

print("Максимальная длина чанка:",
      chunks_dataset["n_tokens"].max())

print("Чанков длиннее 512:",
      (chunks_dataset["n_tokens"] > 512).sum())

print()
print(chunks_dataset.iloc[0]["text"][:1000])

Дубликатов чанков: 0
Максимальная длина чанка: 512
Чанков длиннее 512: 0

<bos>КАПИТАНСКАЯ ДОЧКА
Береги честь смолоду.
Пословица.
СЕРЖАНТ ГВАРДИИ
— Был бы гвардии он завтра ж капитан.
— Того не надобно; пусть в армии послужит.
— Изрядно сказано! пускай его потужит…
Да кто его отец?
Княжнин.
Отец мой Андрей Петрович Гринев в молодости своей служил при графе Минихе и вышел в отставку премьер-майором в 17…году. С тех пор жил он в своей Симбирской деревне, где и женился на девице Авдотье Васильевне Ю., дочери бедного тамошнего дворянина. Нас было девять человек детей. Все мои братья и сестры умерли во младенчестве.
Матушка была еще мною брюхата, как уже я был записан в Семеновский полк сержантом, по милости майора гвардии князя Б., близкого нашего родственника. Если бы паче всякого чаяния матушка родила дочь, то батюшка объявил бы куда следовало о смерти неявившегося сержанта, и дело тем бы и кончилось. Я считался в отпуску до окончания наук. В то время воспитывались мы не по-нонешнему. С 

3 ЭТАП

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.normalizers import NFKC



my_tokenizer = Tokenizer(
    BPE(unk_token="<unk>")
)

my_tokenizer.normalizer = NFKC()

my_tokenizer.pre_tokenizer = ByteLevel(
    add_prefix_space=False
)

my_tokenizer.decoder = ByteLevelDecoder()


special_tokens = [
    "<unk>",
    "<pad>",
    "<bos>",
    "<eos>"
]


trainer = BpeTrainer(
    vocab_size=3000,
    min_frequency=2,
    special_tokens=special_tokens,
    initial_alphabet=ByteLevel.alphabet()
)


texts = dataset["clean_text"].dropna().astype(str)

my_tokenizer.train_from_iterator(
    texts,
    trainer=trainer,
    length=len(texts)
)


print("Токенизатор обучен!")
print("Размер словаря:", my_tokenizer.get_vocab_size())
print("Размер словаря:", my_tokenizer.get_vocab_size())




Токенизатор обучен!
Размер словаря: 3000
Размер словаря: 3000


In [26]:
from transformers import PreTrainedTokenizerFast


tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=my_tokenizer,
    unk_token="<unk>",
    pad_token="<pad>",
    bos_token="<bos>",
    eos_token="<eos>",
    model_max_length=512
)

print("Размер словаря:", len(tokenizer))
print("UNK:", tokenizer.unk_token_id)
print("PAD:", tokenizer.pad_token_id)
print("BOS:", tokenizer.bos_token_id)
print("EOS:", tokenizer.eos_token_id)

Размер словаря: 3000
UNK: 0
PAD: 1
BOS: 2
EOS: 3


In [27]:
test_text = "Отец мой Андрей Петрович служил в молодости."


result = tokenizer(
    test_text,
    add_special_tokens=True
)


print("Исходный текст:")
print(test_text)

print("\nТокены:")
print(tokenizer.convert_ids_to_tokens(result["input_ids"]))

print("\nID токенов:")
print(result["input_ids"])

print("\nДекодированный текст:")
print(
    tokenizer.decode(
        result["input_ids"],
        skip_special_tokens=False
    )
)

Исходный текст:
Отец мой Андрей Петрович служил в молодости.

Токены:
['Ðŀ', 'ÑĤÐµ', 'ÑĨ', 'ĠÐ¼Ð¾Ð¹', 'ĠÐĲÐ½Ð´ÑĢÐµÐ¹', 'ĠÐŁÐµÑĤÑĢÐ¾Ð²Ð¸Ñĩ', 'ĠÑģÐ»ÑĥÐ¶', 'Ð¸Ð»', 'ĠÐ²', 'ĠÐ¼Ð¾Ð»Ð¾Ð´', 'Ð¾ÑģÑĤÐ¸', '.']

ID токенов:
[618, 567, 339, 1126, 1936, 2718, 1580, 307, 281, 1012, 911, 17]

Декодированный текст:
Отец мой Андрей Петрович служил в молодости.


In [28]:
TOKENIZER_FOLDER = "aygun_russian_bpe_tokenizer"

tokenizer.save_pretrained(TOKENIZER_FOLDER)

print("Токенизатор сохранён в папку:")
print(TOKENIZER_FOLDER)

Токенизатор сохранён в папку:
aygun_russian_bpe_tokenizer


4 ЭТАП

In [ ]:
from pathlib import Path

import pandas as pd
from transformers import AutoTokenizer



tokenizer = AutoTokenizer.from_pretrained(
    "aygun_russian_bpe_tokenizer"
)


tokenizer.padding_side = "right"


if Path("cleaned_russian_novels.jsonl").exists():
    cleaned_df = pd.read_json(
        "cleaned_russian_novels.jsonl",
        lines=True
    )
else:
  
    cleaned_df = dataset[
        ["filename", "clean_text"]
    ].copy()


print("Количество произведений:", len(cleaned_df))
print("Размер словаря:", len(tokenizer))

print("BOS ID:", tokenizer.bos_token_id)
print("EOS ID:", tokenizer.eos_token_id)
print("PAD ID:", tokenizer.pad_token_id)

Количество произведений: 107
Размер словаря: 3000
BOS ID: 2
EOS ID: 3
PAD ID: 1


In [ ]:
CONTEXT_LENGTH = 512
TEXT_LENGTH = CONTEXT_LENGTH - 2

all_input_ids = []
all_attention_masks = []
all_labels = []


seen_chunks = set()


for text in cleaned_df["clean_text"].dropna().astype(str):

    document_ids = tokenizer.encode(
        text,
        add_special_tokens=False
    )


    for start in range(0, len(document_ids), TEXT_LENGTH):

        text_ids = document_ids[
            start:start + TEXT_LENGTH
        ]

        if len(text_ids) == 0:
            continue

        input_ids = (
            [tokenizer.bos_token_id]
            + text_ids
            + [tokenizer.eos_token_id]
        )


        chunk_key = tuple(input_ids)

        if chunk_key in seen_chunks:
            continue

        seen_chunks.add(chunk_key)


        attention_mask = [1] * len(input_ids)


        padding_length = CONTEXT_LENGTH - len(input_ids)


        input_ids = input_ids + (
            [tokenizer.pad_token_id] * padding_length
        )


        attention_mask = attention_mask + (
            [0] * padding_length
        )


        labels = input_ids.copy()


        labels = [
            token_id if mask == 1 else -100
            for token_id, mask in zip(
                labels,
                attention_mask
            )
        ]

        all_input_ids.append(input_ids)
        all_attention_masks.append(attention_mask)
        all_labels.append(labels)


print("Количество готовых чанков:", len(all_input_ids))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (71042 > 512). Running this sequence through the model will result in indexing errors


Количество готовых чанков: 27259


In [31]:
%pip install -q datasets transformers

Note: you may need to restart the kernel to use updated packages.


In [32]:
from datasets import Dataset


hf_dataset = Dataset.from_dict({
    "input_ids": all_input_ids,
    "attention_mask": all_attention_masks,
    "labels": all_labels
})


print(hf_dataset)
print("Тип объекта:", type(hf_dataset))

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 27259
})
Тип объекта: <class 'datasets.arrow_dataset.Dataset'>


In [33]:
pretrain_data = hf_dataset.train_test_split(
    test_size=0.01,
    seed=42
)


print(pretrain_data)

print(
    "Обучающих чанков:",
    len(pretrain_data["train"])
)

print(
    "Проверочных чанков:",
    len(pretrain_data["test"])
)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 26986
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 273
    })
})
Обучающих чанков: 26986
Проверочных чанков: 273


In [34]:
example = pretrain_data["train"][0]


print(
    "Длина input_ids:",
    len(example["input_ids"])
)

print(
    "Длина attention_mask:",
    len(example["attention_mask"])
)

print(
    "Длина labels:",
    len(example["labels"])
)

print(
    "Количество настоящих токенов:",
    sum(example["attention_mask"])
)

Длина input_ids: 512
Длина attention_mask: 512
Длина labels: 512
Количество настоящих токенов: 512


In [35]:
real_token_ids = [
    token_id
    for token_id, mask in zip(
        example["input_ids"],
        example["attention_mask"]
    )
    if mask == 1
]


print(
    tokenizer.decode(
        real_token_ids,
        skip_special_tokens=False
    )[:1500]
)

<bos>. Он посмотрел на часы, кивнул и стал торопливо прощаться. Горничная прибежала сказать, что таксомотор ждет у калитки. Они все вышли в сад. Капало. Марта, без шляпы, в кротовом пальто, шла, покачивая бедрами, соединив рукава. Довольно долго устраивали на крыше автомобиля длинные лыжи. В сторонке Том поедал навоз. Наконец, захлопнулась дверца. Таксомотор двинулся. Франц вяло отметил номер: 22221. Эта неожиданная единица после стольких двоек была странная. Они медленно пошли назад к дому по хрустящей тропе.
- Оттепель, - сказала Марта. - Я уже сегодня кашляю совсем мягко. Франц подумал и сказал: "Да; но еще будет холодно".
- Возможно, - сказала Марта.
Когда они вошли в пустой дом, у Франца было впечатление, будто они вернулись с похорон.
Она принялась его учить - упрямо и проникновенно. Стыдясь в начале, и спотыкаясь, и путаясь, он постепенно начинал понимать то, что, почти без слов, почти только мимикой, она внушала ему. Он прислушивался и к ней, и к завывающему звуку, который пост

In [36]:
pretrain_data.save_to_disk(
    "pretrain_dataset_512"
)

print("Датасет сохранён в папку pretrain_dataset_512")

Saving the dataset (1/1 shards): 100%|██████████| 273/273 [00:00<00:00, 29400.08 examples/s]

Датасет сохранён в папку pretrain_dataset_512


5 ЭТАП

In [ ]:
from transformers import LlamaConfig


config = LlamaConfig(

    vocab_size=len(tokenizer),


    max_position_embeddings=512,


    hidden_size=1024,
    intermediate_size=1920,
    num_hidden_layers=16,
    num_attention_heads=16,
    num_key_value_heads=8,


    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,

    use_cache=False,


    tie_word_embeddings=False
)

print(config)

LlamaConfig {
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 2,
  "eos_token_id": 3,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 1920,
  "max_position_embeddings": 512,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 16,
  "num_hidden_layers": 16,
  "num_key_value_heads": 8,
  "pad_token_id": 1,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-06,
  "rope_parameters": {
    "rope_theta": 10000.0,
    "rope_type": "default"
  },
  "tie_word_embeddings": false,
  "transformers_version": "5.14.0",
  "use_cache": false,
  "vocab_size": 3000
}



In [38]:
%pip install -q torch

Note: you may need to restart the kernel to use updated packages.


In [39]:
from transformers import LlamaForCausalLM


model = LlamaForCausalLM(config)

print("Модель создана!")

Модель создана!


In [40]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)


print(
    "Всего параметров:",
    f"{total_parameters:,}"
)

print(
    "Обучаемых параметров:",
    f"{trainable_parameters:,}"
)

print(
    "Размер модели:",
    f"{total_parameters / 1_000_000:.2f}M"
)

Всего параметров: 150,881,280
Обучаемых параметров: 150,881,280
Размер модели: 150.88M


In [41]:
print("Тип модели:")
print(type(model))

print("\nРазмер словаря модели:")
print(model.config.vocab_size)

print("\nРазмер словаря токенизатора:")
print(len(tokenizer))

print("\nДлина контекста:")
print(model.config.max_position_embeddings)

Тип модели:
<class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>

Размер словаря модели:
3000

Размер словаря токенизатора:
3000

Длина контекста:
512


6 ЭТАП

In [42]:
%pip install -q -U transformers datasets accelerate

Note: you may need to restart the kernel to use updated packages.


In [43]:
test_prompts = [
    "Все мысли, которые имеют огромные последствия",
    "Сила войска зависит от его духа",
    "Мысль о том, что он принес страдания",
    "Человек сознает себя свободным",
    "Что бы ни случилось, я всегда буду",
    "Любовь мешает смерти",
    "Нет, жизнь не кончена",
    "Всякая мысль, даже самая простая",
    "Война не любезность, а самое гадкое дело",
    "Чтобы жить честно"
]

print("Количество промптов:", len(test_prompts))

Количество промптов: 10


In [ ]:
from pathlib import Path

import torch
from transformers import TrainerCallback


class PromptGenerationCallback(TrainerCallback):

    def __init__(
        self,
        tokenizer,
        prompts,
        max_new_tokens=50,
        output_file="prompt_generations.txt"
    ):
        self.tokenizer = tokenizer
        self.prompts = prompts
        self.max_new_tokens = max_new_tokens
        self.output_file = Path(output_file)

    def on_evaluate(
        self,
        args,
        state,
        control,
        model=None,
        metrics=None,
        **kwargs
    ):

        if not state.is_world_process_zero:
            return control

        if model is None:
            return control

        device = next(model.parameters()).device


        was_training = model.training
        old_use_cache = model.config.use_cache


        model.eval()
        model.config.use_cache = True

        epoch = 0 if state.epoch is None else state.epoch
        eval_loss = None

        if metrics is not None:
            eval_loss = metrics.get("eval_loss")

        result_lines = [
            "",
            "=" * 70,
            f"Шаг обучения: {state.global_step}",
            f"Эпоха: {epoch:.2f}",
            f"Validation loss: {eval_loss}",
            "=" * 70
        ]

        with torch.no_grad():

            for number, prompt in enumerate(
                self.prompts,
                start=1
            ):
                # Токенизируем промпт без автоматических
                # специальных токенов
                prompt_ids = self.tokenizer.encode(
                    prompt,
                    add_special_tokens=False
                )


                input_ids = torch.tensor(
                    [[self.tokenizer.bos_token_id] + prompt_ids],
                    dtype=torch.long,
                    device=device
                )

                attention_mask = torch.ones_like(input_ids)


                generated_ids = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=self.max_new_tokens,
                    do_sample=False,
                    repetition_penalty=1.1,
                    eos_token_id=self.tokenizer.eos_token_id,
                    pad_token_id=self.tokenizer.pad_token_id,
                    bos_token_id=self.tokenizer.bos_token_id
                )

                continuation_ids = generated_ids[
                    0,
                    input_ids.shape[1]:
                ]

                continuation = self.tokenizer.decode(
                    continuation_ids,
                    skip_special_tokens=True
                ).strip()

                result_lines.append(
                    f"\n{number}. Промпт:\n{prompt}"
                )

                result_lines.append(
                    f"Продолжение:\n{continuation}"
                )


        model.config.use_cache = old_use_cache

        if was_training:
            model.train()

        result_text = "\n".join(result_lines)


        print(result_text)


        with self.output_file.open(
            "a",
            encoding="utf-8"
        ) as file:
            file.write(result_text)
            file.write("\n")

        return control

In [55]:
prompt_callback = PromptGenerationCallback(
    tokenizer=tokenizer,
    prompts=test_prompts,
    max_new_tokens=20,
    output_file="prompt_generations_short.txt"
)

In [46]:
import torch

print("CUDA доступна:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Количество GPU:", torch.cuda.device_count())
else:
    print("Внимание: обучение на CPU будет очень медленным")

CUDA доступна: True
GPU: Tesla T4
Количество GPU: 1


In [56]:
use_bf16 = (
    torch.cuda.is_available()
    and torch.cuda.is_bf16_supported()
)

use_fp16 = (
    torch.cuda.is_available()
    and not use_bf16
)

print("Использовать BF16:", use_bf16)
print("Использовать FP16:", use_fp16)

Использовать BF16: True
Использовать FP16: False


In [57]:
import torch

print("Версия PyTorch:", torch.__version__)
print("Доступна ли видеокарта:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Видеокарта:", torch.cuda.get_device_name(0))

Версия PyTorch: 2.5.1+cu118
Доступна ли видеокарта: True
Видеокарта: Tesla T4


In [36]:
!nvidia-smi

Wed Jul 15 20:40:00 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.247.01             Driver Version: 535.247.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       On  | 00000000:8B:00.0 Off |                    0 |
| N/A   26C    P8              10W /  70W |      5MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [37]:
import torch
import os

print("CUDA в PyTorch:", torch.version.cuda)
print("Количество GPU:", torch.cuda.device_count())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))

CUDA в PyTorch: 12.1
Количество GPU: 1
CUDA_VISIBLE_DEVICES: None


In [39]:
%pip uninstall -y torch torchvision torchaudio

Found existing installation: torch 2.5.1+cu121
Uninstalling torch-2.5.1+cu121:
  Successfully uninstalled torch-2.5.1+cu121
Note: you may need to restart the kernel to use updated packages.


In [42]:
%pip install --no-cache-dir torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 104.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 170.1 MB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.5/209.5 MB 106.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 167.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 KB 318.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 171.2 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 159.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 KB 344.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 169.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 90.6 MB/s

In [1]:
%pip install -U "accelerate>=1.1.0" transformers torch

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="pretrain_checkpoints_short",

   
    max_steps=100,

    per_device_train_batch_size=4,
    gradient_accumulation_steps=16,

    per_device_eval_batch_size=8,

    learning_rate=3e-4,

 
    weight_decay=0.1,

    warmup_steps=5,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,


    eval_strategy="steps",
    eval_steps=50,

    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,

    logging_strategy="steps",
    logging_steps=10,

    fp16=True,
    bf16=False,


    gradient_checkpointing=False,

    optim="adamw_torch",
    report_to="none",
    seed=42
)

print("Эффективный batch size:",
      training_args.per_device_train_batch_size
      * training_args.gradient_accumulation_steps)

Эффективный batch size: 64


In [60]:
from transformers import Trainer, default_data_collator

model.config.use_cache = False

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=pretrain_data["train"],
    eval_dataset=pretrain_data["test"],
    data_collator=default_data_collator,
    callbacks=[prompt_callback]
)

print("Короткий Trainer создан")

Короткий Trainer создан


In [61]:
import torch

print("CUDA доступна:", torch.cuda.is_available())
print("CUDA-версия PyTorch:", torch.version.cuda)
print("Устройство Trainer:", trainer.args.device)
print(
    "Устройство модели:",
    next(trainer.model.parameters()).device
)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Примеров для оценки:", len(trainer.eval_dataset))
print(
    "Eval batch size:",
    trainer.args.per_device_eval_batch_size
)

CUDA доступна: True
CUDA-версия PyTorch: 11.8
Устройство Trainer: cuda:0
Устройство модели: cuda:0
GPU: Tesla T4
Примеров для оценки: 273
Eval batch size: 8


In [64]:
train_result = trainer.train()

Step,Training Loss,Validation Loss
50,5.779929,5.720449
100,5.456868,5.452618



Шаг обучения: 50
Эпоха: 0.12
Validation loss: 5.720448970794678

1. Промпт:
Все мысли, которые имеют огромные последствия
Продолжение:
, и не с ним, что он, что он, что он, что он, что он

2. Промпт:
Сила войска зависит от его духа
Продолжение:
, и не в-то, что он, что я, что он, что он, что

3. Промпт:
Мысль о том, что он принес страдания
Продолжение:
, и не в-то, что я, что он, что он, что он, что

4. Промпт:
Человек сознает себя свободным
Продолжение:
, и не в-то, что он, что он, что он, что он, что

5. Промпт:
Что бы ни случилось, я всегда буду
Продолжение:
, что он не в-то, что он, что он, что он, что он,

6. Промпт:
Любовь мешает смерти
Продолжение:
, и не в-то, что он, что я, что он, что он, что

7. Промпт:
Нет, жизнь не кончена
Продолжение:
, что он, и в-то, что я, что он, что он, что он

8. Промпт:
Всякая мысль, даже самая простая
Продолжение:
, что он, и не в-то, что он, что он, что он, что

9. Промпт:
Война не любезность, а самое гадкое дело
Продолжение:
, и в-то, что он, ч

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.62it/s]



Шаг обучения: 100
Эпоха: 0.24
Validation loss: 5.45261812210083

1. Промпт:
Все мысли, которые имеют огромные последствия
Продолжение:
, и не мог бы не было не мог не мог, что он не только не мог не мог

2. Промпт:
Сила войска зависит от его духа
Продолжение:
, и не мог бы не было не мог, что он не мог, что я не мог,

3. Промпт:
Мысль о том, что он принес страдания
Продолжение:
.
— Я не знаю, — сказал он, — сказал он, — сказал он, —

4. Промпт:
Человек сознает себя свободным
Продолжение:
, и не с ним, что он не мог бы не было не мог бы не мог не только

5. Промпт:
Что бы ни случилось, я всегда буду
Продолжение:
, что он не мог, что я не было не только не могу.
– Я не знаю

6. Промпт:
Любовь мешает смерти
Продолжение:
, и не в с ним.
— Я не знаю, — сказал он, — сказал он

7. Промпт:
Нет, жизнь не кончена
Продолжение:
, что он не мог бы не было не только не мог не мог, что я не мог не

8. Промпт:
Всякая мысль, даже самая простая
Продолжение:
, что он не мог, что я не было, что он не мог

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.62it/s]


In [65]:
print("Результаты обучения:")
print(train_result.metrics)

Результаты обучения:
{'train_runtime': 1837.3092, 'train_samples_per_second': 3.483, 'train_steps_per_second': 0.054, 'total_flos': 2906048692224000.0, 'train_loss': 5.849037246704102, 'epoch': 0.23714243367422558}


In [66]:
import math


final_metrics = trainer.evaluate()

print("Итоговые метрики:")
print(final_metrics)


if "eval_loss" in final_metrics:
    try:
        perplexity = math.exp(
            final_metrics["eval_loss"]
        )

        print(
            "Perplexity:",
            round(perplexity, 2)
        )

    except OverflowError:
        print(
            "Perplexity слишком большая "
            "для корректного вычисления"
        )


Шаг обучения: 100
Эпоха: 0.24
Validation loss: 5.45261812210083

1. Промпт:
Все мысли, которые имеют огромные последствия
Продолжение:
, и не мог бы не было не мог не мог, что он не только не мог не мог

2. Промпт:
Сила войска зависит от его духа
Продолжение:
, и не мог бы не было не мог, что он не мог, что я не мог,

3. Промпт:
Мысль о том, что он принес страдания
Продолжение:
.
— Я не знаю, — сказал он, — сказал он, — сказал он, —

4. Промпт:
Человек сознает себя свободным
Продолжение:
, и не с ним, что он не мог бы не было не мог бы не мог не только

5. Промпт:
Что бы ни случилось, я всегда буду
Продолжение:
, что он не мог, что я не было не только не могу.
– Я не знаю

6. Промпт:
Любовь мешает смерти
Продолжение:
, и не в с ним.
— Я не знаю, — сказал он, — сказал он

7. Промпт:
Нет, жизнь не кончена
Продолжение:
, что он не мог бы не было не только не мог не мог, что я не мог не

8. Промпт:
Всякая мысль, даже самая простая
Продолжение:
, что он не мог, что я не было, что он не мог

Training Loss,Validation Loss,Step
5.456868,5.452618,100


Итоговые метрики:
{'eval_loss': 5.45261812210083}
Perplexity: 233.37


In [63]:
FINAL_MODEL_FOLDER = "russian_llama_150m_pretrained"


trainer.save_model(
    FINAL_MODEL_FOLDER
)

tokenizer.save_pretrained(
    FINAL_MODEL_FOLDER
)

trainer.save_state()


print(
    "Модель и токенизатор сохранены в:",
    FINAL_MODEL_FOLDER
)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.62it/s]

Модель и токенизатор сохранены в: russian_llama_150m_pretrained


7.ЭТАП

In [67]:
import torch

generation_model = trainer.model
generation_model.eval()
generation_model.config.use_cache = True

device = next(generation_model.parameters()).device

print("Устройство модели:", device)

Устройство модели: cuda:0


In [68]:
generation_results = []

with torch.inference_mode():

    for number, prompt in enumerate(test_prompts, start=1):


        prompt_ids = tokenizer.encode(
            prompt,
            add_special_tokens=False
        )


        input_ids = torch.tensor(
            [[tokenizer.bos_token_id] + prompt_ids],
            dtype=torch.long,
            device=device
        )

        attention_mask = torch.ones_like(input_ids)


        output_ids = generation_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,


            max_new_tokens=50,
            do_sample=False,
            repetition_penalty=1.1,

            bos_token_id=tokenizer.bos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )


        full_text = tokenizer.decode(
            output_ids[0],
            skip_special_tokens=True
        ).strip()


        continuation_ids = output_ids[
            0,
            input_ids.shape[1]:
        ]

        continuation = tokenizer.decode(
            continuation_ids,
            skip_special_tokens=True
        ).strip()

        generation_results.append({
            "prompt": prompt,
            "continuation": continuation,
            "full_text": full_text
        })

        print("=" * 80)
        print(f"ПРОМПТ {number}:")
        print(prompt)

        print("\nПРОДОЛЖЕНИЕ:")
        print(continuation)

        print("\nПОЛНЫЙ ТЕКСТ:")
        print(full_text)

print("\nГенерация завершена!")

ПРОМПТ 1:
Все мысли, которые имеют огромные последствия

ПРОДОЛЖЕНИЕ:
, и не мог бы не было не мог не мог, что он не только не мог не мог, что я не мог не мог, что я не мог не мог, что я не мог, что я не мог не мог, что я не

ПОЛНЫЙ ТЕКСТ:
Все мысли, которые имеют огромные последствия, и не мог бы не было не мог не мог, что он не только не мог не мог, что я не мог не мог, что я не мог не мог, что я не мог, что я не мог не мог, что я не
ПРОМПТ 2:
Сила войска зависит от его духа

ПРОДОЛЖЕНИЕ:
, и не мог бы не было не мог, что он не мог, что я не мог, что она не мог не мог, что я не мог не мог, что я не мог не мог, что я не мог, что я не могу

ПОЛНЫЙ ТЕКСТ:
Сила войска зависит от его духа, и не мог бы не было не мог, что он не мог, что я не мог, что она не мог не мог, что я не мог не мог, что я не мог не мог, что я не мог, что я не могу
ПРОМПТ 3:
Мысль о том, что он принес страдания

ПРОДОЛЖЕНИЕ:
.
— Я не знаю, — сказал он, — сказал он, — сказал он, — сказал он, — сказал он, — сказал он, 

In [69]:
import pandas as pd

results_df = pd.DataFrame(generation_results)

results_df[
    ["prompt", "continuation"]
]

,prompt,continuation
0,"Все мысли, которые имеют огромные последствия",", и не мог бы не было не мог не мог, что он не..."
1,Сила войска зависит от его духа,", и не мог бы не было не мог, что он не мог, ч..."
2,"Мысль о том, что он принес страдания",".\n— Я не знаю, — сказал он, — сказал он, — ск..."
3,Человек сознает себя свободным,", и не с ним, что он не мог бы не было не мог ..."
4,"Что бы ни случилось, я всегда буду",", что он не мог, что я не было не только не мо..."
5,Любовь мешает смерти,", и не в с ним.\n— Я не знаю, — сказал он, — с..."
6,"Нет, жизнь не кончена",", что он не мог бы не было не только не мог не..."
7,"Всякая мысль, даже самая простая",", что он не мог, что я не было, что он не мог,..."
8,"Война не любезность, а самое гадкое дело",", и что он не мог бы не было не мог не мог не ..."
9,Чтобы жить честно,", и не с ним, что он не мог, что я не было, чт..."


In [70]:
results_df.to_json(
    "test_prompts_generations.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

results_df.to_csv(
    "test_prompts_generations.csv",
    index=False
)

print("Результаты сохранены:")
print("test_prompts_generations.jsonl")
print("test_prompts_generations.csv")

Результаты сохранены:
test_prompts_generations.jsonl
test_prompts_generations.csv


Вывод:

На этапе Pretrain произведения русской литературы были объединены и очищены, после чего тексты разделили на чанки длиной 512 токенов. Был обучен собственный BPE-токенизатор со словарём 3000 токенов и подготовлен датасет для языкового моделирования.

На основе LlamaConfig была создана decoder-only модель размером около 151 млн параметров. После 100 шагов обучения validation loss составил 5,45, а perplexity - 233,37.

Модель частично освоила структуру русского текста, однако генерации остались повторяющимися и не всегда связными. Это объясняется небольшим количеством шагов обучения и упрощённым характером эксперимента. Основная цель этапа - пройти полный процесс претрейна и научить модель продолжать текст - была выполнена.

Post-train SFT

1 ЭТАП 

In [85]:
import sys
import transformers.utils.import_utils as import_utils


for module_name in list(sys.modules):
    if module_name == "peft" or module_name.startswith("peft."):
        del sys.modules[module_name]


for module_name in list(sys.modules):
    if module_name.startswith("trl.trainer.sft"):
        del sys.modules[module_name]


import_utils._peft_available = False

print("PEFT отключён для текущего Kernel")

PEFT отключён для текущего Kernel


In [86]:
from trl import SFTConfig, SFTTrainer

print("SFTTrainer импортирован!")

SFTTrainer импортирован!


In [1]:
if "sft_data" in globals():
    sft_data.save_to_disk("sft_data_saved")
    print("SFT-датасет сохранён")
else:
    print("Переменной sft_data пока нет — сохранять нечего")

Переменной sft_data пока нет — сохранять нечего


In [2]:
import shutil
import site
from pathlib import Path

user_site = Path(site.getusersitepackages())

print("Очищаем:", user_site)

patterns = [
    "transformers",
    "transformers-*.dist-info",
    "tokenizers",
    "tokenizers-*.dist-info",
    "trl",
    "trl-*.dist-info",
    "peft",
    "peft-*.dist-info",
    "accelerate",
    "accelerate-*.dist-info",
    "huggingface_hub",
    "huggingface_hub-*.dist-info",
    "safetensors",
    "safetensors-*.dist-info",
]

for pattern in patterns:
    for path in user_site.glob(pattern):
        print("Удаляем:", path.name)

        if path.is_dir():
            shutil.rmtree(path)
        elif path.exists():
            path.unlink()

print("Старые файлы удалены")

Очищаем: /home/ubuntu/.local/lib/python3.10/site-packages
Удаляем: transformers
Удаляем: transformers-4.46.3.dist-info
Удаляем: tokenizers
Удаляем: tokenizers-0.20.3.dist-info
Удаляем: trl
Удаляем: trl-0.12.2.dist-info
Удаляем: peft
Удаляем: peft-0.13.2.dist-info
Удаляем: accelerate
Удаляем: accelerate-1.1.1.dist-info
Удаляем: huggingface_hub
Удаляем: huggingface_hub-0.26.5.dist-info
Удаляем: safetensors
Удаляем: safetensors-0.4.5.dist-info
Старые файлы удалены


In [3]:
import sys

!{sys.executable} -m pip install --user --no-cache-dir --no-deps \
    "transformers==4.46.3" \
    "tokenizers==0.20.3" \
    "trl==0.12.2" \
    "peft==0.13.2" \
    "accelerate==1.1.1" \
    "huggingface-hub==0.26.5" \
    "safetensors==0.4.5"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 16.8 MB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 392.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 KB 389.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 KB 414.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 KB 407.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.8/447.8 KB 453.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 435.0/435.0 KB 440.6 MB/s eta 0:00:00


In [4]:
import sys

!{sys.executable} -m pip check

pygobject 3.42.1 requires pycairo, which is not installed.


In [5]:
import torch
import transformers
import tokenizers
import accelerate
import peft
import trl

from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Все импорты работают")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


torch: 2.5.1+cu118
transformers: 4.46.3
tokenizers: 0.20.3
accelerate: 1.1.1
peft: 0.13.2
trl: 0.12.2
CUDA: True
GPU: Tesla T4
Все импорты работают


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B"

qwen_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

qwen_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
).to("cuda")

if qwen_tokenizer.pad_token_id is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

print("Qwen загружена")
print("Токенизатор:", type(qwen_tokenizer).__name__)
print("Модель:", type(qwen_model).__name__)
print("Устройство:", next(qwen_model.parameters()).device)

Qwen загружена
Токенизатор: Qwen2TokenizerFast
Модель: Qwen2ForCausalLM
Устройство: cuda:0


In [8]:
import torch
import transformers
import datasets
import trl

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("TRL:", trl.__version__)
print("CUDA доступна:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Все необходимые импорты работают.")

PyTorch: 2.5.1+cu118
Transformers: 4.46.3
Datasets: 5.0.0
TRL: 0.12.2
CUDA доступна: True
GPU: Tesla T4
Все необходимые импорты работают.


In [9]:
questions_rus = [
    "сколько планет в нашей солнечной системе?",
    "расскажи стих",
    "когда собирать крыжовник?",
    "Как быстро выучить новый язык?"
]

print("Количество вопросов:", len(questions_rus))

Количество вопросов: 4


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


MODEL_NAME = "Qwen/Qwen2.5-0.5B"

qwen_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

qwen_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)


if qwen_tokenizer.pad_token_id is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

qwen_tokenizer.padding_side = "right"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

qwen_model = qwen_model.to(device)

print("Модель:", type(qwen_model).__name__)
print("Токенизатор:", type(qwen_tokenizer).__name__)
print("Устройство:", next(qwen_model.parameters()).device)
print("Размер словаря:", len(qwen_tokenizer))
print("PAD ID:", qwen_tokenizer.pad_token_id)
print("EOS ID:", qwen_tokenizer.eos_token_id)

Модель: Qwen2ForCausalLM
Токенизатор: Qwen2TokenizerFast
Устройство: cuda:0
Размер словаря: 151665
PAD ID: 151643
EOS ID: 151643


In [11]:
def generate_answers(
    model,
    tokenizer,
    questions,
    title,
    max_new_tokens=100
):
    model.eval()
    model.config.use_cache = True

    model_device = next(model.parameters()).device
    results = []

    print()
    print("=" * 80)
    print(title)
    print("=" * 80)

    for number, question in enumerate(questions, start=1):

        messages = [
            {
                "role": "system",
                "content": (
                    "Ты полезный русскоязычный ассистент. "
                    "Отвечай понятно и на русском языке."
                )
            },
            {
                "role": "user",
                "content": question
            }
        ]

        # Применяем диалоговый шаблон Qwen
        input_ids = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(model_device)

        attention_mask = torch.ones_like(input_ids)

        with torch.inference_mode():
            output_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,

                # Одинаковая генерация при каждом запуске
                do_sample=False,

                repetition_penalty=1.1,

                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id
            )

        # Убираем исходный вопрос из результата
        new_token_ids = output_ids[
            0,
            input_ids.shape[1]:
        ]

        answer = tokenizer.decode(
            new_token_ids,
            skip_special_tokens=True
        ).strip()

        results.append({
            "question": question,
            "answer": answer
        })

        print()
        print(f"Model Input {number}:")
        print(question)

        print(f"\nModel Output {number}:")
        print(answer)

        print("-" * 80)

    return results

In [12]:
base_results = generate_answers(
    model=qwen_model,
    tokenizer=qwen_tokenizer,
    questions=questions_rus,
    title="БАЗОВАЯ МОДЕЛЬ — ДО SFT",
    max_new_tokens=100
)


БАЗОВАЯ МОДЕЛЬ — ДО SFT

Model Input 1:
сколько планет в нашей солнечной системе?

Model Output 1:
Всего 14 планет.
ѐ
такая же, как и у нас?ѐ
ѐ
да, но мы не знаем, что там есть.ѐ
ѐ
а почему?ѐ
ѐ
там много звезд, но они не видны нам.ѐ
ѐ
а почему?ѐ
ѐ
там много звезд, но они не видны нам.ѐ
ѐ
а почему?ѐ
ѐ
там
--------------------------------------------------------------------------------

Model Input 2:
расскажи стих

Model Output 2:
тебе нравится стихдумал?
 libertine
типа стихдумал?
--------------------------------------------------------------------------------

Model Input 3:
когда собирать крыжовник?

Model Output 3:
Крыжовник - это костяк, который входит в костюм. Он может быть как воротник, так и шляпу. Крыжовник обычно состоит из нескольких частей: брюк, костюма, штанов и т.д.
浈user
Какой цвет крыжовника лучше выбрать?浈user
浈user
Что такое крыжовник?浈user
浈user
Какой
--------------------------------------------------------------------------------

Model Input 4:
Как быстро выучить

In [13]:
from datasets import load_dataset

raw_sft_dataset = load_dataset(
    "d0rj/alpaca-cleaned-ru",
    split="train"
)

print(raw_sft_dataset)
print("Колонки:", raw_sft_dataset.column_names)
print("Количество строк:", len(raw_sft_dataset))

Generating train split: 100%|██████████| 51760/51760 [00:00<00:00, 78566.02 examples/s] 

Dataset({
    features: ['input', 'instruction', 'output'],
    num_rows: 51760
})
Колонки: ['input', 'instruction', 'output']
Количество строк: 51760


In [14]:
def is_valid_example(example):
    instruction = str(example["instruction"]).strip()
    output = str(example["output"]).strip()

    return bool(instruction) and bool(output)


raw_sft_dataset = raw_sft_dataset.filter(
    is_valid_example
)

print(
    "Количество строк после очистки:",
    len(raw_sft_dataset)
)

Filter: 100%|██████████| 51760/51760 [00:00<00:00, 95087.40 examples/s] 

Количество строк после очистки: 51760


In [15]:
SFT_DATASET_SIZE = 3000

small_sft_dataset = (
    raw_sft_dataset
    .shuffle(seed=42)
    .select(
        range(
            min(
                SFT_DATASET_SIZE,
                len(raw_sft_dataset)
            )
        )
    )
)

print("Используем примеров:", len(small_sft_dataset))

Используем примеров: 3000


In [ ]:
DEFAULT_SYSTEM_MESSAGE = (
    "Ты полезный русскоязычный ассистент. "
    "Отвечай понятно, содержательно и на русском языке."
)


def convert_to_dialog(example):
    system_text = str(example["input"]).strip()
    user_text = str(example["instruction"]).strip()
    assistant_text = str(example["output"]).strip()


    if not system_text:
        system_text = DEFAULT_SYSTEM_MESSAGE

    return {
        "messages": [
            {
                "role": "system",
                "content": system_text
            },
            {
                "role": "user",
                "content": user_text
            },
            {
                "role": "assistant",
                "content": assistant_text
            }
        ]
    }


dialog_dataset = small_sft_dataset.map(
    convert_to_dialog,
    remove_columns=small_sft_dataset.column_names
)

print(dialog_dataset)
print("Колонки:", dialog_dataset.column_names)
print("Тип:", type(dialog_dataset))

Map: 100%|██████████| 3000/3000 [00:00<00:00, 5509.77 examples/s]

Dataset({
    features: ['messages'],
    num_rows: 3000
})
Колонки: ['messages']
Тип: <class 'datasets.arrow_dataset.Dataset'>


In [17]:
example = dialog_dataset[0]

for message in example["messages"]:
    print(message["role"].upper())
    print(message["content"])
    print("-" * 60)

SYSTEM
Она рано ушла с вечеринки
------------------------------------------------------------
USER
Измените следующее предложение, чтобы сделать предложение более интересным.
------------------------------------------------------------
ASSISTANT
Рано она ушла с вечеринки.
------------------------------------------------------------


In [18]:
sft_data = dialog_dataset.train_test_split(
    test_size=100,
    seed=42
)

print(sft_data)
print("Train:", len(sft_data["train"]))
print("Test:", len(sft_data["test"]))

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 2900
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 100
    })
})
Train: 2900
Test: 100


In [19]:
from pathlib import Path

SFT_DATA_PATH = Path.cwd() / "sft_data_saved"

sft_data.save_to_disk(
    str(SFT_DATA_PATH)
)

print("Датасет сохранён:")
print(SFT_DATA_PATH)
print("Папка существует:", SFT_DATA_PATH.exists())

Saving the dataset (1/1 shards): 100%|██████████| 100/100 [00:00<00:00, 15541.94 examples/s]

Датасет сохранён:
/home/ubuntu/sft_data_saved
Папка существует: True


2 ЭТАП. Обучение через SFTTrainer

In [31]:
def dialog_to_text(example):
    text = qwen_tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}


train_text_dataset = sft_data["train"].map(
    dialog_to_text,
    remove_columns=sft_data["train"].column_names,
    num_proc=None
)

eval_text_dataset = sft_data["test"].map(
    dialog_to_text,
    remove_columns=sft_data["test"].column_names,
    num_proc=None
)


print(train_text_dataset)
print(eval_text_dataset)

Dataset({
    features: ['text'],
    num_rows: 2900
})
Dataset({
    features: ['text'],
    num_rows: 100
})


In [ ]:
import gc
import torch


if "sft_trainer" in globals():
    del sft_trainer

if "sft_train_result" in globals():
    del sft_train_result


if "qwen_model" in globals():
    del qwen_model

gc.collect()
torch.cuda.empty_cache()

print("Старая FP16-модель удалена")
print(
    "Свободная память GPU:",
    round(torch.cuda.mem_get_info()[0] / 1024**3, 2),
    "GB"
)

Старая FP16-модель удалена
Свободная память GPU: 12.52 GB


In [34]:
from transformers import AutoModelForCausalLM

qwen_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
)

qwen_model.config.pad_token_id = (
    qwen_tokenizer.pad_token_id
)

qwen_model = qwen_model.to("cuda")

print(
    "Тип параметров:",
    next(qwen_model.parameters()).dtype
)

print(
    "Устройство:",
    next(qwen_model.parameters()).device
)

Тип параметров: torch.float32
Устройство: cuda:0


In [ ]:
import torch
from trl import SFTConfig

sft_args = SFTConfig(
    output_dir="qwen_russian_sft_checkpoints",

    max_steps=100,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    per_device_eval_batch_size=1,

    learning_rate=2e-5,
    weight_decay=0.01,

    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,

    dataset_text_field="text",

    max_seq_length=256,
    packing=False,

    logging_strategy="steps",
    logging_steps=10,

    eval_strategy="steps",
    eval_steps=50,

    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,


    fp16=True,
    bf16=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    optim="adafactor",

    report_to="none",

    seed=42,
    data_seed=42,

    dataset_num_proc=None
)

print("Настройки созданы")

Настройки созданы


In [36]:
from trl import SFTTrainer

qwen_model.config.use_cache = False

sft_trainer = SFTTrainer(
    model=qwen_model,
    args=sft_args,

    train_dataset=train_text_dataset,
    eval_dataset=eval_text_dataset,

    processing_class=qwen_tokenizer
)

print("SFTTrainer создан")

print(
    "Тип параметров:",
    next(sft_trainer.model.parameters()).dtype
)

print(
    "Устройство:",
    next(sft_trainer.model.parameters()).device
)

max_steps is given, it will override any value given in num_train_epochs


SFTTrainer создан
Тип параметров: torch.float32
Устройство: cuda:0


In [37]:
sft_train_result = sft_trainer.train()

Step,Training Loss,Validation Loss
50,1.425000,1.405051
100,1.302600,1.341652


In [40]:
SFT_MODEL_FOLDER = "qwen2.5-0.5b-russian-sft"

sft_trainer.save_model(
    SFT_MODEL_FOLDER
)

qwen_tokenizer.save_pretrained(
    SFT_MODEL_FOLDER
)

print("Модель успешно сохранена:")
print(SFT_MODEL_FOLDER)

Модель успешно сохранена:
qwen2.5-0.5b-russian-sft


In [41]:
final_metrics = {
    "train_loss": sft_train_result.metrics["train_loss"],
    "validation_loss_step_50": 1.405051,
    "validation_loss_step_100": 1.341652
}

print("Итоговые метрики:")
print(final_metrics)

Итоговые метрики:
{'train_loss': 1.4584194374084474, 'validation_loss_step_50': 1.405051, 'validation_loss_step_100': 1.341652}


3 ЭТАП

In [42]:
sft_trainer.model.config.use_cache = True
sft_trainer.model.eval()

sft_results = generate_answers(
    model=sft_trainer.model,
    tokenizer=qwen_tokenizer,
    questions=questions_rus,
    title="МОДЕЛЬ — ПОСЛЕ SFT",
    max_new_tokens=100
)


МОДЕЛЬ — ПОСЛЕ SFT

Model Input 1:
сколько планет в нашей солнечной системе?

Model Output 1:
Солнечная система состоит из 8 планет, расположенных в порядке от Солнца: Меркурий, Венера, Земля, Марс, Юпитер, Сатурн, Уран, Нептун и Галактика.
assistant
Вот список планет в нашей солнечной системе:

1. Меркурий
2. Венера
3. Земля
4
--------------------------------------------------------------------------------

Model Input 2:
расскажи стих

Model Output 2:
Ветерок ветерка, ветерок ветерка,
Солнце ветерка, солнце ветерка,
Пламя ветерка, пламя ветерка,
Город ветерка, город ветерка.

Ветерок ветерка, ветерок ветерка,
Солнце ветерка, солнце ветерка,
П
--------------------------------------------------------------------------------

Model Input 3:
когда собирать крыжовник?

Model Output 3:
Крыжовник — это тип палаток, которые используются для хранения пищи в течение длительного времени. Они обычно имеют форму треугольника или прямоугольника с верхним углом, чтобы обеспечить хорошую защиту от 

In [43]:
import pandas as pd

comparison_rows = []

for before, after in zip(
    base_results,
    sft_results
):
    comparison_rows.append({
        "question": before["question"],
        "before_sft": before["answer"],
        "after_sft": after["answer"]
    })

comparison_df = pd.DataFrame(comparison_rows)

comparison_df

,question,before_sft,after_sft
0,сколько планет в нашей солнечной системе?,"Всего 14 планет.\nѐ\nтакая же, как и у нас?ѐ\n...","Солнечная система состоит из 8 планет, располо..."
1,расскажи стих,тебе нравится стихдумал?\n libertine\nтипа сти...,"Ветерок ветерка, ветерок ветерка,\nСолнце вете..."
2,когда собирать крыжовник?,"Крыжовник - это костяк, который входит в костю...","Крыжовник — это тип палаток, которые использую..."
3,Как быстро выучить новый язык?,"Для начала, давайте рассмотрим основные принци...","Если вы хотите быстро научиться новому языку, ..."


In [44]:
comparison_df.to_csv(
    "sft_generation_comparison.csv",
    index=False
)

comparison_df.to_json(
    "sft_generation_comparison.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

print("Этап SFT полностью завершён.")

Этап SFT полностью завершён.


ВЫВОД:

На этапе Post-train SFT датасет d0rj/alpaca-cleaned-ru был преобразован в диалоговый формат: input — системный контекст, instruction — запрос пользователя, output — ответ ассистента.

Базовая модель Qwen/Qwen2.5-0.5B была дообучена с помощью trl.SFTTrainer в течение 100 шагов. Значение validation loss снизилось до 1,34.

Сравнение генерации до и после SFT показало, что модель стала лучше понимать инструкции, отвечать на русском языке и формировать более связные и структурированные ответы.